# Final Merchant Recommendations

This notebook presents the final merchant recommendations for BNPL onboarding.

The ranking methodology was developed and validated in `ranking_summary.ipynb`. This notebook focuses on the final decision outputs: validating the reliability of the selected merchants, presenting the final Top 100 merchants, and identifying the Top 10 merchants within each industry segment.

The final ranking combines:

- Merchant value
- Customer reach/scale
- Revenue growth
- Revenue stability
- Market / regional context
- Fraud-risk safety

The baseline final score uses the validated weighting framework developed in the ranking methodology notebook.

CustomerScore represents customer reach/scale via `unique_consumers`; repeat usage is not directly included. The ranking is a decision-support heuristic, not a statistically optimal prediction model.

## 1. Load Final Ranking Base

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

cwd = Path.cwd().resolve()

if (cwd / "member5_ranking").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "member5_ranking").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the repository root containing member5_ranking."
    )

RANKING_PATH = (
    PROJECT_ROOT
    / "member5_ranking"
    / "results"
    / "ranking_analysis_base.csv"
)

ranking_df = pd.read_csv(RANKING_PATH)

print("Shape:", ranking_df.shape)
print("Unique merchants:", ranking_df["merchant_abn"].nunique())

ranking_df.head()

Shape: (4026, 89)
Unique merchants: 4026


,merchant_abn,merchant_name,merchant_category,merchant_pricing_level,merchant_take_rate_pct,has_merchant_master_record,total_transactions,total_revenue,avg_transaction_value,unique_consumers,...,final_score_30c_70knn,fraud_5_score,fraud_5_rank,fraud_10_score,fraud_10_rank,fraud_15_score,fraud_15_rank,fraud_weight_best_rank,fraud_weight_worst_rank,fraud_weight_rank_range
0,10023283211,Felis Limited,"furniture, home furnishings and equipment shop...",E,0.18,True,3261,703277.711451,215.663205,3032,...,58.094870,57.787813,1531,57.603741,1498,57.419669,1444,1444,1531,87
1,10142254217,Arcu Ac Orci Corporation,"cable, satellite, and other pay television and...",B,4.22,True,3036,118356.146073,38.984238,2849,...,59.509701,60.945617,1338,60.000731,1339,59.055845,1349,1338,1349,11
2,10165489824,Nunc Sed Company,"jewelry, watch, clock, and silverware shops",B,4.40,True,5,56180.473857,11236.094771,5,...,29.597010,26.228697,3506,29.819132,3362,33.409567,3171,3171,3506,335
3,10187291046,Ultricies Dignissim Lacus Foundation,"watch, clock, and jewelry repair shops",B,3.29,True,336,39693.730387,118.136102,335,...,41.563857,42.138348,2534,42.865861,2499,43.593375,2448,2448,2534,86
4,10192359162,Enim Condimentum PC,"music shops - musical instruments, pianos, and...",A,6.33,True,385,177980.505456,462.287027,383,...,57.650820,58.669756,1478,56.979380,1529,55.289004,1596,1478,1596,118


## 2. Final Ranking Reliability Audit

Before presenting the final recommendations, the Top 100 merchants are checked for three potential reliability concerns:

1. whether growth estimates are based on insufficient transaction history,
2. whether external Census / SEIFA / ATO coverage is adequate, and
3. how strongly KNN similarity supports the fraud-risk estimates, using confidence and out-of-distribution diagnostics.

These checks do not directly change the ranking. They are used to assess how much confidence should be placed in the final recommendations.

In [2]:
final_top100_df = (
    ranking_df
    .sort_values("final_rank")
    .head(100)
    .copy()
)

print("Final Top 100 merchants:", len(final_top100_df))

Final Top 100 merchants: 100


In [3]:
growth_reliability = (
    final_top100_df["low_sample_growth_estimate"]
    .value_counts(dropna=False)
    .rename_axis("low_sample_growth_estimate")
    .to_frame("merchant_count")
)

growth_reliability

,merchant_count
low_sample_growth_estimate,
False,100


In [4]:
low_sample_count = final_top100_df["low_sample_growth_estimate"].fillna(False).sum()

print("Low-sample growth merchants:", low_sample_count)
print("Share of Top 100:", low_sample_count / 100)

Low-sample growth merchants: 0
Share of Top 100: 0.0


In [5]:
external_coverage_summary = (
    final_top100_df[
        "regional_data_coverage_rate_all_sources_by_count"
    ]
    .describe()
)

external_coverage_summary

count    100.000000
mean       0.807613
std        0.003106
min        0.800461
25%        0.805886
50%        0.807655
75%        0.809167
max        0.815884
Name: regional_data_coverage_rate_all_sources_by_count, dtype: float64

In [6]:
coverage_checks = pd.Series({
    "coverage_below_90pct":
        (
            final_top100_df[
                "regional_data_coverage_rate_all_sources_by_count"
            ] < 0.90
        ).sum(),

    "coverage_below_80pct":
        (
            final_top100_df[
                "regional_data_coverage_rate_all_sources_by_count"
            ] < 0.80
        ).sum(),

    "coverage_below_70pct":
        (
            final_top100_df[
                "regional_data_coverage_rate_all_sources_by_count"
            ] < 0.70
        ).sum(),
})

coverage_checks

coverage_below_90pct    100
coverage_below_80pct      0
coverage_below_70pct      0
dtype: int64

In [7]:
overall_external_coverage = (
    ranking_df[
        "regional_data_coverage_rate_all_sources_by_count"
    ]
    .describe()
)

top100_external_coverage = (
    final_top100_df[
        "regional_data_coverage_rate_all_sources_by_count"
    ]
    .describe()
)

coverage_comparison = pd.DataFrame({
    "all_eligible_merchants": overall_external_coverage,
    "final_top_100": top100_external_coverage,
})

coverage_comparison

,all_eligible_merchants,final_top_100
count,4026.000000,100.000000
mean,0.806718,0.807613
std,0.054179,0.003106
min,0.000000,0.800461
25%,0.796875,0.805886
50%,0.807871,0.807655
75%,0.819749,0.809167
max,1.000000,0.815884


In [8]:
overall_mean_coverage = ranking_df[
    "regional_data_coverage_rate_all_sources_by_count"
].mean()

top100_mean_coverage = final_top100_df[
    "regional_data_coverage_rate_all_sources_by_count"
].mean()

print("All eligible merchants mean coverage:", overall_mean_coverage)
print("Top 100 mean coverage:", top100_mean_coverage)
print("Difference:", top100_mean_coverage - overall_mean_coverage)

All eligible merchants mean coverage: 0.806718209745659
Top 100 mean coverage: 0.8076128858879301
Difference: 0.0008946761422711225


External-data coverage is highly consistent among the final Top 100 merchants. Their mean joint coverage across Census, SEIFA, and ATO sources is 80.76%, which is very close to the 80.67% average across all eligible merchants.

This suggests that the final recommendations are not disproportionately driven by merchants with poor external-data coverage.

In [9]:
knn_diagnostic_columns = ["knn_confidence_score", "knn_out_of_distribution"]
if ranking_df[knn_diagnostic_columns].isna().any().any():
    raise ValueError("KNN reliability diagnostics are missing for eligible merchants.")
if not ranking_df["knn_out_of_distribution"].isin([True, False]).all():
    raise ValueError("KNN out-of-distribution flags must be boolean.")

knn_reliability_summary = pd.Series({
    "top100_ood_count": int(final_top100_df["knn_out_of_distribution"].sum()),
    "top100_ood_share": final_top100_df["knn_out_of_distribution"].mean(),
    "top100_mean_knn_confidence": final_top100_df["knn_confidence_score"].mean(),
    "all_eligible_mean_knn_confidence": ranking_df["knn_confidence_score"].mean(),
    "top20_ood_count": int(final_top100_df.head(20)["knn_out_of_distribution"].sum()),
}, name="value")

knn_reliability_summary

top100_ood_count                    12.000000
top100_ood_share                     0.120000
top100_mean_knn_confidence           0.441967
all_eligible_mean_knn_confidence     0.193619
top20_ood_count                      0.000000
Name: value, dtype: float64

### Reliability Audit Interpretation

None of the final Top 100 merchants is flagged as having a low-sample growth estimate, indicating that the growth component is not being driven by merchants with insufficient transaction history.

The Top 100 has average joint Census, SEIFA, and ATO coverage of 80.76%, compared with 80.67% across all eligible merchants. Coverage is consistent with the wider eligible pool.

Among the final Top 100, 12 merchants (12%) are flagged as out-of-distribution. Mean KNN confidence is approximately 0.442, compared with 0.194 across all eligible merchants. None of the Top 20 merchants is out-of-distribution.

Top-ranked recommendations generally have stronger KNN support than the wider eligible pool. KNN merchant fraud scores are relative risk signals, not precise fraud probabilities. The 12 out-of-distribution merchants should be interpreted with greater caution, not automatically excluded; these diagnostics do not alter the locked ranking.

## 3. Locked Final Scoring Framework

Following feature construction, business-weight sensitivity analysis, fraud integration, fraud-weight sensitivity analysis, and reliability checks, the final ranking framework is fixed before producing the recommendation lists.

The final score is:

\[
\text{Final Score}
=
0.90 \times \text{Business Score}
+
0.10 \times \text{Fraud-Risk Safety Score}
\]

where:

\[
\text{Business Score}
=
0.30 \times \text{Value}
+
0.25 \times \text{Customer Reach/Scale}
+
0.20 \times \text{Growth}
+
0.15 \times \text{Stability}
+
0.10 \times \text{Market Context}
\]

This is equivalent to the following effective final weights:

- **Merchant Value: 27%**
- **Customer Reach/Scale: 22.5%**
- **Growth: 18%**
- **Stability: 13.5%**
- **Market / Regional Context: 9%**
- **Fraud-Risk Safety: 10%**

The weighting framework prioritises directly observed commercial performance while retaining supporting signals for future growth, consistency, socioeconomic context, and fraud risk.

Reliability checks indicate that:

- none of the final Top 100 merchants has a low-sample growth estimate;
- external Census, SEIFA, and ATO coverage among the Top 100 is consistent with the wider eligible merchant pool; and
- Top 100 mean KNN confidence is approximately 0.442 versus 0.194 across the eligible pool; 12 Top 100 merchants are out-of-distribution, while none of the Top 20 is.

The scoring framework is therefore treated as fixed for the final recommendation stage.

The existing `final_score = 0.90 * balanced_score + 0.10 * risk_safety_score` is consumed unchanged from the ranking base. The 50/50 consumer/KNN fraud composition is a balanced baseline, not a statistically optimal weight. Fraud safety remains a 10% baseline adjustment.

In [10]:
required_final_columns = [
    "final_score",
    "final_rank",
    "balanced_score",
    "risk_safety_score",
    "value_score",
    "customer_score",
    "growth_score",
    "stability_score",
    "market_score",
]

missing_final_columns = [
    col for col in required_final_columns
    if col not in ranking_df.columns
]

print("Missing required final columns:", missing_final_columns)

if missing_final_columns:
    raise ValueError(f"Missing required final columns: {missing_final_columns}")
if ranking_df[required_final_columns].isna().any().any():
    raise ValueError("The locked ranking contains missing scores or ranks.")
np.testing.assert_allclose(
    ranking_df["final_score"],
    0.90 * ranking_df["balanced_score"] + 0.10 * ranking_df["risk_safety_score"],
)
np.testing.assert_allclose(
    ranking_df["balanced_score"],
    0.30 * ranking_df["value_score"]
    + 0.25 * ranking_df["customer_score"]
    + 0.20 * ranking_df["growth_score"]
    + 0.15 * ranking_df["stability_score"]
    + 0.10 * ranking_df["market_score"],
)
np.testing.assert_array_equal(
    ranking_df["final_rank"],
    ranking_df["final_score"].rank(method="min", ascending=False).astype(int),
)

Missing required final columns: []


## 4. Final Top 100 Merchants

The final Top 100 merchants are selected using the locked final scoring framework.

These merchants represent the strongest overall onboarding candidates after considering commercial value, customer reach/scale, growth, stability, market context, and fraud-risk safety.

In [11]:
final_top100 = final_top100_df.copy()

final_top100[
    [
        "final_rank",
        "merchant_abn",
        "merchant_name",
        "merchant_category",
        "final_score",
        "balanced_score",
        "risk_safety_score",
        "value_score",
        "customer_score",
        "growth_score",
        "stability_score",
        "market_score",
    ]
].head(20)

,final_rank,merchant_abn,merchant_name,merchant_category,final_score,balanced_score,risk_safety_score,value_score,customer_score,growth_score,stability_score,market_score
1301,1,38090089066,Interdum Feugiat Sed Inc.,"furniture, home furnishings and equipment shop...",85.084485,86.800764,69.637978,98.683557,99.254844,61.607365,93.928838,59.711873
3614,2,90568944804,Diam Eu Dolor LLC,tent and awning shops,84.834617,87.168863,63.826405,99.230005,93.169399,75.167952,82.483205,67.014406
3369,3,84703983173,Amet Consulting,"computer programming , data processing, and in...",84.630599,85.765124,74.419880,96.572280,99.006458,68.847972,89.524757,48.435171
3443,4,86578477987,Leo In Consulting,"watch, clock, and jewelry repair shops",84.539430,85.650047,74.543876,99.925484,99.975161,55.113212,95.172929,53.800298
803,5,27326652377,Tellus Aenean Corporation,"music shops - musical instruments, pianos, and...",84.364009,86.253316,67.360254,99.403875,89.195231,75.267479,86.165713,61.549925
1401,6,40515428545,Elit Sed Consequat Associates,artist supply and craft shops,84.276448,84.659810,80.826198,99.652260,94.585196,50.684250,94.700174,67.759563
348,7,17488304283,Posuere Cubilia Curae Corporation,"cable, satellite, and other pay television and...",84.273831,85.080227,77.016270,97.913562,98.559364,64.170192,93.480965,42.101341
1798,8,49212265466,Auctor Company,"florists supplies, nursery stock, and flowers",84.246497,85.552242,72.494798,99.056135,98.782911,60.014929,94.575765,49.503229
3453,9,86772484982,Posuere Cubilia Curae LLC,"digital goods: books, movies, music",84.237758,85.785250,70.310332,95.032290,95.355191,59.442647,98.283155,68.057625
3881,10,96680767841,Ornare Limited,motor vehicle supplies and new parts,84.216013,85.723766,70.646239,99.850969,98.435171,63.772083,90.097039,48.907104


In [12]:
top20_inspection = final_top100[
    [
        "final_rank",
        "merchant_abn",
        "merchant_name",
        "merchant_category",
        "final_score",
        "balanced_score",
        "risk_safety_score",
        "value_score",
        "customer_score",
        "growth_score",
        "stability_score",
        "market_score",
        "knn_confidence_score",
        "knn_out_of_distribution",
    ]
].head(20).copy()

top20_inspection.to_csv(
    PROJECT_ROOT
    / "member5_ranking"
    / "results"
    / "final_top20_inspection.csv",
    index=False
)

top20_inspection

,final_rank,merchant_abn,merchant_name,merchant_category,final_score,balanced_score,risk_safety_score,value_score,customer_score,growth_score,stability_score,market_score,knn_confidence_score,knn_out_of_distribution
1301,1,38090089066,Interdum Feugiat Sed Inc.,"furniture, home furnishings and equipment shop...",85.084485,86.800764,69.637978,98.683557,99.254844,61.607365,93.928838,59.711873,0.918033,False
3614,2,90568944804,Diam Eu Dolor LLC,tent and awning shops,84.834617,87.168863,63.826405,99.230005,93.169399,75.167952,82.483205,67.014406,0.327869,False
3369,3,84703983173,Amet Consulting,"computer programming , data processing, and in...",84.630599,85.765124,74.419880,96.572280,99.006458,68.847972,89.524757,48.435171,0.459016,False
3443,4,86578477987,Leo In Consulting,"watch, clock, and jewelry repair shops",84.539430,85.650047,74.543876,99.925484,99.975161,55.113212,95.172929,53.800298,0.639344,False
803,5,27326652377,Tellus Aenean Corporation,"music shops - musical instruments, pianos, and...",84.364009,86.253316,67.360254,99.403875,89.195231,75.267479,86.165713,61.549925,0.245902,False
1401,6,40515428545,Elit Sed Consequat Associates,artist supply and craft shops,84.276448,84.659810,80.826198,99.652260,94.585196,50.684250,94.700174,67.759563,0.557377,False
348,7,17488304283,Posuere Cubilia Curae Corporation,"cable, satellite, and other pay television and...",84.273831,85.080227,77.016270,97.913562,98.559364,64.170192,93.480965,42.101341,0.377049,False
1798,8,49212265466,Auctor Company,"florists supplies, nursery stock, and flowers",84.246497,85.552242,72.494798,99.056135,98.782911,60.014929,94.575765,49.503229,0.885246,False
3453,9,86772484982,Posuere Cubilia Curae LLC,"digital goods: books, movies, music",84.237758,85.785250,70.310332,95.032290,95.355191,59.442647,98.283155,68.057625,0.065574,False
3881,10,96680767841,Ornare Limited,motor vehicle supplies and new parts,84.216013,85.723766,70.646239,99.850969,98.435171,63.772083,90.097039,48.907104,0.967213,False


In [13]:
final_top20 = (
    final_top100
    .head(20)
    .copy()
)

print("Final Top 20 merchants:", len(final_top20))

Final Top 20 merchants: 20


In [14]:
top20_score_summary = (
    final_top20[
        [
            "final_score",
            "balanced_score",
            "risk_safety_score",
            "value_score",
            "customer_score",
            "growth_score",
            "stability_score",
            "market_score",
        ]
    ]
    .describe()
    .T
)

top20_score_summary

,count,mean,std,min,25%,50%,75%,max
final_score,20.0,84.094967,0.441355,83.590999,83.716550,84.049209,84.298339,85.084485
balanced_score,20.0,85.433539,0.675385,84.659810,84.953734,85.226394,85.734106,87.168863
risk_safety_score,20.0,72.047821,3.524726,63.826405,70.428115,71.424651,73.926679,80.826198
value_score,20.0,98.550671,1.547925,95.032290,97.901143,99.143070,99.683308,99.975161
customer_score,20.0,97.414928,2.872712,89.195231,95.737084,98.621461,99.316940,99.975161
growth_score,20.0,60.528738,6.553414,50.684250,55.449117,59.641702,62.839015,75.267479
stability_score,20.0,91.487932,3.679577,82.483205,89.879323,91.714357,94.090570,98.283155
market_score,20.0,56.856682,9.036517,42.101341,49.354198,54.992548,65.021113,72.205663


In [15]:
score_dimensions = [
    "value_score",
    "customer_score",
    "growth_score",
    "stability_score",
    "market_score",
    "risk_safety_score",
]

top20_weakest_dimension = final_top20[
    ["final_rank", "merchant_name"] + score_dimensions
].copy()

top20_weakest_dimension["weakest_dimension"] = (
    top20_weakest_dimension[score_dimensions].idxmin(axis=1)
)

top20_weakest_dimension["weakest_score"] = (
    top20_weakest_dimension[score_dimensions].min(axis=1)
)

top20_weakest_dimension[
    [
        "final_rank",
        "merchant_name",
        "weakest_dimension",
        "weakest_score",
    ]
]

,final_rank,merchant_name,weakest_dimension,weakest_score
1301,1,Interdum Feugiat Sed Inc.,market_score,59.711873
3614,2,Diam Eu Dolor LLC,risk_safety_score,63.826405
3369,3,Amet Consulting,market_score,48.435171
3443,4,Leo In Consulting,market_score,53.800298
803,5,Tellus Aenean Corporation,market_score,61.549925
1401,6,Elit Sed Consequat Associates,growth_score,50.684250
348,7,Posuere Cubilia Curae Corporation,market_score,42.101341
1798,8,Auctor Company,market_score,49.503229
3453,9,Posuere Cubilia Curae LLC,growth_score,59.442647
3881,10,Ornare Limited,market_score,48.907104


### Top 20 Profile Check

The Top 20 merchants are consistently strong in the core commercial dimensions of merchant value, customer reach/scale, and revenue stability.

Variation is mainly concentrated in growth, market context, and fraud-risk safety. No Top 20 merchant exhibits an extreme weakness in the core commercial dimensions, suggesting that the highest-ranked merchants are supported by broad business strength rather than a single dominant metric.

In [16]:
final_top100.to_csv(
    PROJECT_ROOT
    / "member5_ranking"
    / "results"
    / "final_top_100.csv",
    index=False
)

print("Final Top 100 exported successfully.")

Final Top 100 exported successfully.


## 5. Segment-Level Top 10 Recommendations

To complement the overall Top 100 ranking, merchants are also compared within their broader industry segments.

The same locked final score is used for all segments. This preserves consistency across the recommendation framework while allowing strong merchants in smaller or structurally different industries to be identified.

For each segment, the Top 10 merchants are selected according to their final score.

In [17]:
MAPPING_PATH = (
    PROJECT_ROOT
    / "member3_industry_growth"
    / "results"
    / "category_to_group_mapping.csv"
)

segment_mapping = pd.read_csv(MAPPING_PATH)

print("Mapping shape:", segment_mapping.shape)
segment_mapping.head()

Mapping shape: (25, 2)


,merchant_category,industry_group
0,"antique shops - sales, repairs, and restoratio...","Art, Gifts, Jewellery & Fashion"
1,art dealers and galleries,"Art, Gifts, Jewellery & Fashion"
2,artist supply and craft shops,"Creative, Books & Leisure"
3,bicycle shops - sales and service,"Mobility, Health & Specialist Services"
4,"books, periodicals, and newspapers","Creative, Books & Leisure"


In [18]:
ranking_with_segment = ranking_df.merge(
    segment_mapping,
    on="merchant_category",
    how="left",
    validate="many_to_one",
)

print("Before merge:", len(ranking_df))
print("After merge:", len(ranking_with_segment))
print(
    "Missing industry group:",
    ranking_with_segment["industry_group"].isna().sum()
)

ranking_with_segment["industry_group"].value_counts()

Before merge: 4026
After merge: 4026
Missing industry group: 0


industry_group
Digital, Technology & Communications      867
Home, Garden & Living                     827
Creative, Books & Leisure                 827
Mobility, Health & Specialist Services    806
Art, Gifts, Jewellery & Fashion           699
Name: count, dtype: int64

In [19]:
segment_top10 = (
    ranking_with_segment
    .sort_values(
        ["industry_group", "final_score"],
        ascending=[True, False]
    )
    .groupby("industry_group", group_keys=False)
    .head(10)
    .copy()
)

segment_top10["segment_rank"] = (
    segment_top10
    .groupby("industry_group")["final_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

segment_top10[
    [
        "industry_group",
        "segment_rank",
        "final_rank",
        "merchant_abn",
        "merchant_name",
        "merchant_category",
        "final_score",
        "balanced_score",
        "risk_safety_score",
    ]
].sort_values(
    ["industry_group", "segment_rank"]
)

,industry_group,segment_rank,final_rank,merchant_abn,merchant_name,merchant_category,final_score,balanced_score,risk_safety_score
3796,"Art, Gifts, Jewellery & Fashion",1,17,94493496784,Dictum Phasellus In Institute,"gift, card, novelty, and souvenir shops",83.649721,84.910756,72.300401
2638,"Art, Gifts, Jewellery & Fashion",2,19,68559320474,Aliquam Auctor Associates,"antique shops - sales, repairs, and restoratio...",83.614261,84.968060,71.430070
2293,"Art, Gifts, Jewellery & Fashion",3,32,60956456424,Ultricies Dignissim LLP,"gift, card, novelty, and souvenir shops",83.113148,84.398617,71.543925
3957,"Art, Gifts, Jewellery & Fashion",4,34,98314397036,Lobortis Augue Industries,"gift, card, novelty, and souvenir shops",83.084571,84.544309,69.946926
1637,"Art, Gifts, Jewellery & Fashion",5,38,45629217853,Lacus Consulting,"gift, card, novelty, and souvenir shops",82.992625,83.962811,74.260950
3139,"Art, Gifts, Jewellery & Fashion",6,42,79417999332,Phasellus At Company,"gift, card, novelty, and souvenir shops",82.838218,84.712139,65.972931
65,"Art, Gifts, Jewellery & Fashion",7,51,11439466003,Blandit At LLC,shoe shops,82.429340,83.065984,76.699538
3753,"Art, Gifts, Jewellery & Fashion",8,52,93558142492,Dolor Quisque Inc.,shoe shops,82.423475,82.680867,80.106944
2512,"Art, Gifts, Jewellery & Fashion",9,54,66079287213,Hendrerit Corporation,"gift, card, novelty, and souvenir shops",82.390683,84.162466,66.444635
125,"Art, Gifts, Jewellery & Fashion",10,56,12870663624,Vestibulum Ut Eros Corporation,"gift, card, novelty, and souvenir shops",82.317415,82.768985,78.253287


In [20]:
print("Total selected merchants:", len(segment_top10))

segment_top10.groupby(
    "industry_group"
).size()

Total selected merchants: 50


industry_group
Art, Gifts, Jewellery & Fashion           10
Creative, Books & Leisure                 10
Digital, Technology & Communications      10
Home, Garden & Living                     10
Mobility, Health & Specialist Services    10
dtype: int64

In [21]:
print(
    "Unique merchants in segment Top 10:",
    segment_top10["merchant_abn"].nunique()
)

segment_in_overall_top100 = segment_top10["merchant_abn"].isin(final_top100["merchant_abn"])
print("Segment Top 10 merchants also in overall Top 100:", int(segment_in_overall_top100.sum()))

Unique merchants in segment Top 10: 50
Segment Top 10 merchants also in overall Top 100: 50


### Segment-Level Summary

The segment-level recommendations use the same locked final score as the overall ranking.

The summary below compares the five industry groups using the average final score, average fraud-risk safety, and the range of overall ranks represented within each segment Top 10.

In [22]:
segment_summary = (
    segment_top10
    .groupby("industry_group")
    .agg(
        merchants_selected=("merchant_abn", "count"),
        mean_final_score=("final_score", "mean"),
        mean_balanced_score=("balanced_score", "mean"),
        mean_risk_safety_score=("risk_safety_score", "mean"),
        best_overall_rank=("final_rank", "min"),
        worst_overall_rank=("final_rank", "max"),
    )
    .sort_values("mean_final_score", ascending=False)
)

segment_summary

,merchants_selected,mean_final_score,mean_balanced_score,mean_risk_safety_score,best_overall_rank,worst_overall_rank
industry_group,,,,,,
"Home, Garden & Living",10,83.775738,85.186375,71.080007,1,35
"Digital, Technology & Communications",10,83.570059,84.736375,73.073214,3,47
"Mobility, Health & Specialist Services",10,83.536834,84.874692,71.496106,4,49
"Creative, Books & Leisure",10,83.196147,84.488385,71.566003,5,78
"Art, Gifts, Jewellery & Fashion",10,82.885346,84.017500,72.695961,17,56


In [23]:
segment_score_spread = (
    segment_top10
    .groupby("industry_group")["final_score"]
    .agg(["min", "median", "max"])
)

segment_score_spread["score_range"] = (
    segment_score_spread["max"]
    - segment_score_spread["min"]
)

segment_score_spread

,min,median,max,score_range
industry_group,,,,
"Art, Gifts, Jewellery & Fashion",82.317415,82.915422,83.649721,1.332306
"Creative, Books & Leisure",81.787624,83.436903,84.364009,2.576386
"Digital, Technology & Communications",82.509368,83.543028,84.630599,2.121231
"Home, Garden & Living",83.078474,83.657496,85.084485,2.006012
"Mobility, Health & Specialist Services",82.472430,83.567391,84.539430,2.067000


### Segment-Level Interpretation

The five industry segments show broadly similar performance among their Top 10 merchants. Mean final scores range from approximately 82.9 to 83.8, suggesting that the final scoring framework does not strongly favour a single industry group.

Within each segment, the Top 10 scores are also relatively concentrated, with score ranges of approximately 1.3 to 2.6 points.

All 50 segment-level Top 10 merchants also fall within the overall Top 100 ranking. Therefore, the segment-level recommendations do not introduce weaker merchants solely to achieve industry representation. Instead, they provide an industry-specific view of merchants that are already strong overall candidates.

In [24]:
segment_top10.to_csv(
    PROJECT_ROOT
    / "member5_ranking"
    / "results"
    / "segment_top_10.csv",
    index=False
)

segment_summary.to_csv(
    PROJECT_ROOT
    / "member5_ranking"
    / "results"
    / "segment_summary.csv"
)

print("Segment Top 10 and segment summary exported successfully.")

Segment Top 10 and segment summary exported successfully.
